# Module 5: Human-in-the-Loop Validation Breakpoints

In this notebook, we will build a **Human-Approval workflow**.

Workflow:
1. **Draft Node**: Takes a user request and designs a sensitive operations plan (e.g. database migration).
2. **Breakpoint**: We interrupt execution BEFORE running the execution step.
3. **Human Approval**: An operator inspects the state, updates an approval flag, and resumes execution.
4. **Execution Node**: Performs the task if approved.

### Step 1: Initialize Chat Model Connection

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path="../../../langchain/.env")

model = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    model_name="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.3,
)
print("Model client connected!")

Model client connected!


---
## 1. Defining the Graph State

We need to store the task description, the generated execution plan draft, an approval status flag, and a list of action execution logs.

In [4]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ApprovalState(TypedDict):
    task_description: str
    execution_plan: str
    is_approved: bool
    action_logs: list[str]

---
## 2. Defining Nodes

Let's write our graph nodes. Note that `execution_node` checks the `is_approved` boolean flag in the state.

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

def draft_plan_node(state: ApprovalState) -> dict:
    """Node 1: Drafts an operations plan based on the task description."""
    print("--- Node: draft_plan_node ---")
    task = state.get("task_description")
    
    res = model.invoke([
        SystemMessage(content="You are an database administrator. Write a 1-sentence safe database query plan to achieve the user's task."),
        HumanMessage(content=f"Task: {task}")
    ])
    
    # Set default approval to False and save draft
    return {
        "execution_plan": res.content,
        "is_approved": False
    }


def execute_plan_node(state: ApprovalState) -> dict:
    """Node 2: Performs the database operation if approved."""
    print("--- Node: execute_plan_node ---")
    approved = state.get("is_approved", False)
    plan = state.get("execution_plan")
    
    if not approved:
        print("[SECURITY RISK ALERT] Blocked attempt to execute plan without approval!")
        return {"action_logs": ["BLOCKED: Plan lacked operator approval"]}
    
    # Perform action
    print(f"Executing Database SQL Plan: {plan}")
    return {"action_logs": [f"EXECUTED: Query successfully run for plan: {plan}"]}

---
## 3. Compiling the Graph with Breakpoints

We construct the graph, compile it, and register `execute_plan_node` inside `interrupt_before`. This forces the runtime to pause right before the execution step starts.

In [6]:
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(ApprovalState)

builder.add_node("draft_plan", draft_plan_node)
builder.add_node("execute_plan", execute_plan_node)

builder.add_edge(START, "draft_plan")
builder.add_edge("draft_plan", "execute_plan")
builder.add_edge("execute_plan", END)

# Checkpointer is strictly required for interrupts to work!
memory_saver = MemorySaver()

graph = builder.compile(
    checkpointer=memory_saver,
    interrupt_before=["execute_plan"] # Halt before execute_plan runs
)
print("Graph compiled with active breakpoint before 'execute_plan'!")

Graph compiled with active breakpoint before 'execute_plan'!


---
## 4. Executing and Inspecting the Halt

Let's invoke the graph to perform a schema update query.

In [7]:
config = {"configurable": {"thread_id": "db_maintenance_1"}}
initial_input = {"task_description": "Add a column 'middle_name' to the users database table."}

print("Starting execution...")
graph.invoke(initial_input, config=config)

Starting execution...
--- Node: draft_plan_node ---


{'task_description': "Add a column 'middle_name' to the users database table.",
 'execution_plan': 'ALTER TABLE users ADD COLUMNIF NOT EXISTS middle_name VARCHAR(100);',
 'is_approved': False}

Notice that the execution finished immediately after `draft_plan` ran. It did not print anything for `execute_plan`. Let's inspect the graph state to verify it is paused.

In [8]:
snapshot = graph.get_state(config)
print("--- PAUSED STATE SNAPSHOT ---")
print("Next node in queue:     ", snapshot.next)
print("Current Execution Plan:  ", snapshot.values.get("execution_plan"))
print("Approval Status Flag:    ", snapshot.values.get("is_approved"))
print("Action Logs:             ", snapshot.values.get("action_logs"))

--- PAUSED STATE SNAPSHOT ---
Next node in queue:      ('execute_plan',)
Current Execution Plan:   ALTER TABLE users ADD COLUMNIF NOT EXISTS middle_name VARCHAR(100);
Approval Status Flag:     False
Action Logs:              None


As we can see, `snapshot.next` lists `('execute_plan',)` indicating it is paused, and `is_approved` is `False`.

---
## 5. Simulating Human Approval & Resuming

Now we simulate the operator inspecting the plan, updating `is_approved` to `True` using `update_state`, and resuming execution.

In [9]:
print("Updating state: Setting approval flag to True...")
graph.update_state(
    config,
    {"is_approved": True},
    as_node="draft_plan" # Overwrite values as if draft_plan emitted them
)

# Check the updated state values
updated_snapshot = graph.get_state(config)
print("\nUpdated Approval Flag:", updated_snapshot.values.get("is_approved"))

Updating state: Setting approval flag to True...

Updated Approval Flag: True


Now, we resume the execution by calling `invoke()`, passing `None` as the input payload.

In [10]:
print("Resuming execution...")
graph.invoke(None, config=config)

final_snapshot = graph.get_state(config)
print("\n--- POST-RESUMED STATE SNAPSHOT ---")
print("Next node in queue:", final_snapshot.next)
print("Action Logs:       ", final_snapshot.values.get("action_logs"))

Resuming execution...
--- Node: execute_plan_node ---
Executing Database SQL Plan: ALTER TABLE users ADD COLUMNIF NOT EXISTS middle_name VARCHAR(100);

--- POST-RESUMED STATE SNAPSHOT ---
Next node in queue: ()
Action Logs:        ['EXECUTED: Query successfully run for plan: ALTER TABLE users ADD COLUMNIF NOT EXISTS middle_name VARCHAR(100);']


Success! The breakpoint was successfully processed, and after the manual approval update, `execute_plan` ran to completion, generating the final database logs.